In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/default/lakehouse_raw_data/streaming_orders/")

True

In [0]:
from pyspark.sql import Row
from datetime import datetime
import random

data = []

for i in range(1000):

    data.append(
        Row(
            order_id=i,
            customer_id=random.randint(1, 100),
            amount=round(random.uniform(50, 1000), 2),
            status=random.choice(["completed", "pending", "cancelled"]),
            order_time=str(datetime.now())
        )
    )

df = spark.createDataFrame(data)

df.write.mode("append").json(
    "/Volumes/workspace/default/lakehouse_raw_data/streaming_orders/"
)

print("Streaming files generated!")

Streaming files generated!


In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("order_time", StringType(), True)
])

In [0]:
streaming_df = spark.readStream \
    .schema(schema) \
    .json("/Volumes/workspace/default/lakehouse_raw_data/streaming_orders/")

In [0]:
query = streaming_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option(
        "checkpointLocation",
        "/Volumes/workspace/default/lakehouse_raw_data/checkpoints/orders_stream"
    ) \
    .start(
        "/Volumes/workspace/default/lakehouse_raw_data/bronze/streaming_orders"
    )

query.awaitTermination()

In [0]:
display(
    spark.read.format("delta").load(
        "/Volumes/workspace/default/lakehouse_raw_data/bronze/streaming_orders"
    )
)

order_id,customer_id,amount,status,order_time
625,55,590.68,cancelled,2026-05-25 16:59:05.391229
626,24,247.59,cancelled,2026-05-25 16:59:05.391235
627,37,349.07,cancelled,2026-05-25 16:59:05.391242
628,43,640.53,cancelled,2026-05-25 16:59:05.391249
629,89,955.27,cancelled,2026-05-25 16:59:05.391256
630,29,426.37,completed,2026-05-25 16:59:05.391263
631,46,353.51,completed,2026-05-25 16:59:05.391269
632,24,118.77,cancelled,2026-05-25 16:59:05.391276
633,95,227.03,completed,2026-05-25 16:59:05.391284
634,12,387.86,cancelled,2026-05-25 16:59:05.391291


In [0]:
stream_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/lakehouse_raw_data/bronze/streaming_orders"
)

print("Streaming Bronze Count:", stream_df.count())

display(stream_df.limit(20))

Streaming Bronze Count: 6000


order_id,customer_id,amount,status,order_time
625,55,590.68,cancelled,2026-05-25 16:59:05.391229
626,24,247.59,cancelled,2026-05-25 16:59:05.391235
627,37,349.07,cancelled,2026-05-25 16:59:05.391242
628,43,640.53,cancelled,2026-05-25 16:59:05.391249
629,89,955.27,cancelled,2026-05-25 16:59:05.391256
630,29,426.37,completed,2026-05-25 16:59:05.391263
631,46,353.51,completed,2026-05-25 16:59:05.391269
632,24,118.77,cancelled,2026-05-25 16:59:05.391276
633,95,227.03,completed,2026-05-25 16:59:05.391284
634,12,387.86,cancelled,2026-05-25 16:59:05.391291
